In [ ]:
from typing import OrderedDict
with gzip.open(path_to_file, 'rb') as f:
    temp_y=torch.load(f'../../data/resnet18_svhn.pth', map_location='cpu')
    # temp_y= OrderedDict({k:v.cpu() for k,v in pickle.load(f).items()})
y, shape_dict = dict_to_array(temp_y)
vec_slices = _get_vec_slices(shape_dict)

slice_start = 0
slice_length = 0
vec_slices = []
plt.figure(figsize=(12, 5))
percentiles_per_k = []
names = []
for i, (k, v) in enumerate(shape_dict.items()):
    slice_length += int(np.prod(v))
    if 'num_batches_tracked' in k:
        continue

    percentiles_per_k += [np.percentile(temp_y[k], [1,99])]
    names+=[k]
    if i+1 < len(shape_dict):
        curr_l_name = '.'.join(k.split('.')[:-1])
        temp = list(shape_dict.keys())[i+1]
        next_l_name = '.'.join(temp.split('.')[:-1])
        if curr_l_name == next_l_name and len(curr_l_name)>0:
            continue

    vec_slices.append(slice(slice_start, slice_start + slice_length))
    slice_start += slice_length
    slice_length = 0

    # plt.hist(y[vec_slices[-1]], alpha=0.1, bins=100, color='red')
    # plt.xlim(-2, 2)
    percentiles_per_k = np.array(percentiles_per_k).T
    plt.plot(percentiles_per_k[0], label=curr_l_name+'_min')
    plt.plot(percentiles_per_k[1], label=curr_l_name+'_max')
    plt.scatter(np.arange(len(percentiles_per_k[0])),percentiles_per_k[0])
    plt.scatter(np.arange(len(percentiles_per_k[0])),percentiles_per_k[1])
    plt.xticks(np.arange(len(names)), names, rotation=45, ha='right')
    plt.legend()
    plt.show()
    plt.figure(figsize=(12, 5))
    percentiles_per_k = []
    names = []
